<a href="https://colab.research.google.com/github/abhinav9115001-svg/Wildfire---streamlite-app/blob/main/wildfire_detection_and_monitoring_improved_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import rasterio
import os
import random
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import seaborn as sns

print("TF version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

In [ ]:
import kagglehub

path = kagglehub.dataset_download(
    "caffeinatedhighs/sentinel-2-msi-wildfire-data-for-research"
)
print("Dataset path:", path)

# Inspect folder structure
for root, dirs, files in os.walk(path):
    tifs = [f for f in files if f.endswith('.tif')]
    if tifs:
        print(f"  {root}  →  {len(tifs)} files")

In [ ]:
file_paths, labels = [], []

for root, dirs, files in os.walk(path):
    for file in files:
        if file.endswith(".tif"):
            full_path = os.path.join(root, file)
            if "chop_no_firee" in root:      # MUST check longer string first
                file_paths.append(full_path)
                labels.append(0)
            elif "chop_fire" in root:
                file_paths.append(full_path)
                labels.append(1)
            # silently skip anything unrecognised

file_paths = np.array(file_paths)
labels     = np.array(labels)

print(f"Total samples : {len(file_paths)}")
print(f"Fire    (1)   : {np.sum(labels == 1)}")
print(f"No-fire (0)   : {np.sum(labels == 0)}")

# SANITY CHECK — make sure no image is in the wrong folder
assert len(file_paths) == len(labels), "Mismatch!"
assert 0 not in np.unique(labels) or 1 not in np.unique(labels) or \
       set(np.unique(labels)).issubset({0,1}), "Bad labels found!"
print(" Labels look clean")

In [ ]:
# Load and visually verify one fire and one no-fire image
def quick_inspect(path, label_name):
    with rasterio.open(path) as src:
        img = src.read()           # (bands, H, W)
    img_t = np.transpose(img, (1,2,0))  # (H, W, bands)
    img_norm = np.clip(img_t, 0, 10000) / 10000.0

    nir  = img_norm[:, :, 4]
    swir = img_norm[:, :, 5]
    nbr  = (nir - swir) / (nir + swir + 1e-8)

    # Bands 2,1,0 = R,G,B for Sentinel-2 in this dataset
    rgb = img_t[:, :, [2,1,0]].astype(np.float32)

    # contrast stretch (FIX)
    p2, p98 = np.percentile(rgb, (2, 98))
    rgb = np.clip((rgb - p2) / (p98 - p2 + 1e-8), 0, 1)  # boost contrast for display

    print(f"  {label_name} | shape: {img_t.shape} | "
          f"raw range: [{img.min()}, {img.max()}] | "
          f"NBR mean: {nbr.mean():.3f}")

    plt.figure(figsize=(8, 3))
    plt.subplot(1,2,1); plt.imshow(rgb);  plt.title(f"{label_name} — RGB"); plt.axis('off')
    plt.subplot(1,2,2); plt.imshow(nbr, cmap='RdYlGn', vmin=-1, vmax=1)
    plt.title("NBR  (red=burned)"); plt.colorbar(); plt.tight_layout(); plt.show()

fire_paths    = file_paths[labels == 1]
nofire_paths  = file_paths[labels == 0]

print("=== FIRE SAMPLE ===")
quick_inspect(fire_paths[0], "FIRE")

print("=== NO-FIRE SAMPLE ===")
quick_inspect(nofire_paths[0], "NO-FIRE")

In [ ]:
# 70% train | 15% val | 15% test — all stratified
train_val_paths, test_paths, train_val_labels, test_labels = train_test_split(
    file_paths, labels, test_size=0.15, random_state=42, stratify=labels
)
train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_val_paths, train_val_labels,
    test_size=0.176, random_state=42, stratify=train_val_labels
)

print(f"Train : {len(train_paths)}  "
      f"(fire={np.sum(train_labels==1)}, no-fire={np.sum(train_labels==0)})")
print(f"Val   : {len(val_paths)}    "
      f"(fire={np.sum(val_labels==1)},  no-fire={np.sum(val_labels==0)})")
print(f"Test  : {len(test_paths)}   "
      f"(fire={np.sum(test_labels==1)},  no-fire={np.sum(test_labels==0)})")

In [ ]:
def load_image(path):
    def _load(p):
        with rasterio.open(p.decode()) as src:
            img = src.read()
        img = np.transpose(img, (1, 2, 0)).astype(np.float32)
        img = np.clip(img, 0, 10000) / 10000.0   # global Sentinel-2 scale
        return img

    img = tf.numpy_function(_load, [path], tf.float32)
    img.set_shape((512, 512, 6))
    return img

In [ ]:
def create_dataset(paths, lbs, batch_size=8, augment=False):
    ds = tf.data.Dataset.from_tensor_slices((paths, lbs))
    ds = ds.map(lambda x, y: (load_image(x), y),
                num_parallel_calls=tf.data.AUTOTUNE)

    if augment:
        aug = tf.keras.Sequential([
            tf.keras.layers.RandomFlip("horizontal_and_vertical"),
            tf.keras.layers.RandomRotation(0.2),
            tf.keras.layers.RandomZoom(0.1),
            # NO RandomBrightness — corrupts band ratios
        ])
        ds = ds.map(lambda x, y: (aug(x, training=True), y),
                    num_parallel_calls=tf.data.AUTOTUNE)
        ds = ds.shuffle(buffer_size=300, reshuffle_each_iteration=True)

    return ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

BATCH = 8
train_ds = create_dataset(train_paths, train_labels, BATCH, augment=True)
val_ds   = create_dataset(val_paths,   val_labels,   BATCH)
test_ds  = create_dataset(test_paths,  test_labels,  BATCH)

print("Datasets ready ")

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import (GlobalAveragePooling2D, Dense,
                                      Dropout, BatchNormalization,
                                      Input, Conv2D)
from tensorflow.keras.models import Model

# ── Step 1: get pretrained 3-channel weights ──────────────────────────
base_rgb   = EfficientNetB0(weights='imagenet', include_top=False,
                             input_shape=(512, 512, 3))
first_conv = next(l for l in base_rgb.layers if isinstance(l, Conv2D))
orig_w     = first_conv.get_weights()      # [kernel, (bias)]
orig_kernel = orig_w[0]                    # (k, k, 3, filters)

# ── Step 2: build 6-channel model ─────────────────────────────────────
inp        = Input(shape=(512, 512, 6))
base_model = EfficientNetB0(weights=None, include_top=False,
                             input_tensor=inp)

# ── Step 3: adapt kernel 3→6 channels ────────────────────────────────
new_kernel  = np.concatenate([orig_kernel] * 2, axis=2) / 2.0
first_conv6 = next(l for l in base_model.layers if isinstance(l, Conv2D))
new_weights = [new_kernel] + (orig_w[1:] if len(orig_w) > 1 else [])
first_conv6.set_weights(new_weights)
print(f"Weight adapted: 3→6 channels   kernel shape: {new_kernel.shape}")

# ── Step 4: classification head ───────────────────────────────────────
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
x = Dense(64, activation='relu')(x)
x = Dropout(0.3)(x)
output = Dense(1, activation='sigmoid')(x)

model = Model(inputs=base_model.input, outputs=output)
print(f"Model built  Total params: {model.count_params():,}")

In [ ]:
from tensorflow.keras.callbacks import (EarlyStopping, ModelCheckpoint,
                                          ReduceLROnPlateau)

neg  = int(np.sum(train_labels == 0))
pos  = int(np.sum(train_labels == 1))
total = neg + pos
class_weight = {0: (1/neg)*(total/2.0),
                1: (1/pos)*(total/2.0)}
print(f"Class weights — No-fire: {class_weight[0]:.3f}  Fire: {class_weight[1]:.3f}")

def get_callbacks():
    return [
        EarlyStopping(monitor='val_loss', patience=5,
                      restore_best_weights=True, verbose=1),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=3, min_lr=1e-8, verbose=1),
        ModelCheckpoint('best_model.keras', monitor='val_accuracy',
                        save_best_only=True, mode='max', verbose=1),
    ]


In [ ]:
# ── Freeze entire base ────────────────────────────────────────────────
base_model.trainable = False
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print(f"Trainable params (head only): "
      f"{sum(np.prod(v.shape) for v in model.trainable_variables):,}")

print("\n=== PHASE 1: training head only ===")
history1 = model.fit(
    train_ds,
    validation_data=val_ds,          # val — NOT test
    epochs=15,
    callbacks=get_callbacks(),
    class_weight=class_weight
)

print(f"\nPhase 1 done  ")
print(f"Best val accuracy: {max(history1.history['val_accuracy']):.3f}")

In [ ]:
# ── Unfreeze all, keep first 20% frozen for stability ─────────────────
base_model.trainable = True
n = len(base_model.layers)
for layer in base_model.layers[:n // 5]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(5e-6),   # very low LR
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print(f"Trainable params (full): "
      f"{sum(np.prod(v.shape) for v in model.trainable_variables):,}")

print("\n=== PHASE 2: full fine-tuning ===")
history2 = model.fit(
    train_ds,
    validation_data=val_ds,          # val — NOT test
    epochs=30,
    callbacks=get_callbacks(),
    class_weight=class_weight
)

print(f"\nPhase 2 done ")
print(f"Best val accuracy: {max(history2.history['val_accuracy']):.3f}")

In [ ]:
def plot_history(h1, h2):
    acc  = h1.history['accuracy']     + h2.history['accuracy']
    vacc = h1.history['val_accuracy'] + h2.history['val_accuracy']
    loss = h1.history['loss']         + h2.history['loss']
    vloss= h1.history['val_loss']     + h2.history['val_loss']

    ep   = range(1, len(acc)+1)
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(ep, acc,  label='Train acc')
    ax1.plot(ep, vacc, label='Val acc')
    ax1.axvline(len(h1.history['accuracy']), color='gray',
                linestyle='--', label='Phase 2 start')
    ax1.set_title("Accuracy"); ax1.legend(); ax1.set_ylim(0,1)

    ax2.plot(ep, loss,  label='Train loss')
    ax2.plot(ep, vloss, label='Val loss')
    ax2.axvline(len(h1.history['loss']), color='gray',
                linestyle='--', label='Phase 2 start')
    ax2.set_title("Loss"); ax2.legend()

    plt.tight_layout(); plt.show()

plot_history(history1, history2)

In [ ]:
print("=== FINAL EVALUATION ON HELD-OUT TEST SET ===\n")

# Load best saved model
best_model = tf.keras.models.load_model('best_model.keras')
best_model.evaluate(test_ds, verbose=1)

y_pred_probs = best_model.predict(test_ds).flatten()
y_pred = (y_pred_probs > 0.5).astype(int)
y_true = np.concatenate([y for _, y in test_ds], axis=0)

print("\n", classification_report(y_true, y_pred,
                                   target_names=['No-Fire', 'Fire']))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No-Fire','Fire'],
            yticklabels=['No-Fire','Fire'])
plt.title("Confusion Matrix — Test Set")
plt.ylabel("True"); plt.xlabel("Predicted")
plt.tight_layout(); plt.show()

# ROC curve
fpr, tpr, _ = roc_curve(y_true, y_pred_probs)
roc_auc = auc(fpr, tpr)
plt.figure(figsize=(5,4))
plt.plot(fpr, tpr, label=f"AUC = {roc_auc:.3f}", linewidth=2)
plt.plot([0,1],[0,1],'--', color='gray')
plt.xlabel("FPR"); plt.ylabel("TPR")
plt.title("ROC Curve"); plt.legend(); plt.show()

from sklearn.metrics import recall_score, f1_score
print(f"Fire Recall (missed fires): {recall_score(y_true, y_pred):.3f}")
print(f"Fire F1 Score             : {f1_score(y_true, y_pred):.3f}")

In [ ]:
def compute_nbr(img):    # img is (H,W,6), normalised 0-1
    nir  = img[:, :, 4]
    swir = img[:, :, 5]
    return (nir - swir) / (nir + swir + 1e-8)

def get_last_conv_name(model):
    return [l.name for l in model.layers
            if isinstance(l, tf.keras.layers.Conv2D)][-1]

def make_gradcam_heatmap(img_array, mdl):
    layer_name = get_last_conv_name(mdl)
    grad_model = tf.keras.models.Model(
        mdl.inputs,
        [mdl.get_layer(layer_name).output, mdl.output]
    )
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array)
        loss = preds[:, 0]
    grads  = tape.gradient(loss, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0,1,2))
    hmap   = conv_out[0] @ pooled[..., tf.newaxis]
    hmap   = tf.squeeze(hmap)
    hmap   = tf.maximum(hmap, 0)
    hmap   = hmap / (tf.math.reduce_max(hmap) + 1e-8)
    return hmap.numpy()

# Pick a random fire image for the visualisation
sample_path = random.choice(fire_paths)
with rasterio.open(sample_path) as src:
    raw = src.read()
img = np.clip(np.transpose(raw, (1,2,0)).astype(np.float32), 0, 10000) / 10000.0

import cv2
hmap   = make_gradcam_heatmap(np.expand_dims(img, 0), best_model)
hmap   = cv2.resize(hmap, (512,512))
hmap_c = cv2.applyColorMap(np.uint8(255*hmap), cv2.COLORMAP_JET)

rgb = img[:, :, [2,1,0]].astype(np.float32)

# same contrast stretch here
p2, p98 = np.percentile(rgb, (2, 98))
rgb = np.clip((rgb - p2) / (p98 - p2 + 1e-8), 0, 1)   # correct R,G,B indices
rgb8 = (rgb * 255).astype(np.uint8)
overlay = cv2.addWeighted(rgb8, 0.6, hmap_c, 0.4, 0)

nbr = compute_nbr(img)
nbr_norm = (nbr - nbr.min()) / (nbr.max() - nbr.min() + 1e-8)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(rgb);       axes[0].set_title("RGB");               axes[0].axis('off')
axes[1].imshow(overlay);   axes[1].set_title("Grad-CAM Attention"); axes[1].axis('off')
im = axes[2].imshow(nbr_norm, cmap='RdYlGn')
axes[2].set_title("NBR (Burn Index)");  plt.colorbar(im, ax=axes[2])
plt.suptitle(f"Sample: {os.path.basename(sample_path)}", y=1.01)
plt.tight_layout(); plt.show()